In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

W1115 17:08:26.452000 48557 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
events_df = pd.read_csv(DATA_DIR / "extracted_events_v2.csv")

print("Total events:", len(events_df))
print("Total articles:", events_df["article_id"].nunique())

events_df.head()

Total events: 42
Total articles: 3


,article_id,sentence_id,publication_date,original_sentence,event_trigger,action,subject,dependency,head_action,predicate_type,pos_tag,subject_source,object,agent,passive,negation,temporal_entities,location_entities
0,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,began,begin,['Continuous rainfall'],ROOT,begin,main,VBD,direct,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
1,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,causing,cause,['Continuous rainfall'],advcl,begin,subordinate,VBG,inherited_subordinate,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
2,article_01,1,2026-08-14,Continuous rainfall began across Riverton late...,flood,flood,['several streets'],ccomp,cause,subordinate,VB,direct,[],[],False,False,"['August 13', 'early Friday morning']","['Riverton', 'the eastern part of the city']"
3,article_01,2,2026-08-14,The city weather office recorded 118 millimetr...,recorded,record,['The city weather office'],ROOT,record,main,VBD,direct,['118 millimetres'],[],False,False,['twelve hours'],[]
4,article_01,3,2026-08-14,Emergency services received their first major ...,received,receive,['Emergency services'],ROOT,receive,main,VBD,direct,['their first major flooding calls'],[],False,False,"['approximately 5:30 a.m.', 'August 14']",[]


In [3]:
events_df["event_text"] = events_df["original_sentence"].fillna("").astype(str)

events_df[[
    "article_id",
    "event_trigger",
    "action",
    "event_text"
]].head(10)

,article_id,event_trigger,action,event_text
0,article_01,began,begin,Continuous rainfall began across Riverton late...
1,article_01,causing,cause,Continuous rainfall began across Riverton late...
2,article_01,flood,flood,Continuous rainfall began across Riverton late...
3,article_01,recorded,record,The city weather office recorded 118 millimetr...
4,article_01,received,receive,Emergency services received their first major ...
5,article_01,entered,enter,Water entered ground-floor homes in the Rivers...
6,article_01,diverted,divert,Water entered ground-floor homes in the Rivers...
7,article_01,opened,open,The municipal corporation opened two temporary...
8,article_01,moved,move,Officials said that 46 residents had been move...
9,article_01,warned,warn,The weather office warned that further rainfal...


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer loaded successfully!


In [5]:
event_embeddings = model.encode(
    events_df["event_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", event_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (42, 384)


In [6]:
similarity_matrix = cosine_similarity(event_embeddings)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (42, 42)


array([[1.0000001 , 1.0000001 , 1.0000001 , ..., 0.09378735, 0.09378735,
        0.1179496 ],
       [1.0000001 , 1.0000001 , 1.0000001 , ..., 0.09378735, 0.09378735,
        0.1179496 ],
       [1.0000001 , 1.0000001 , 1.0000001 , ..., 0.09378735, 0.09378735,
        0.1179496 ],
       ...,
       [0.09378735, 0.09378735, 0.09378735, ..., 1.0000004 , 1.0000004 ,
        0.02401601],
       [0.09378735, 0.09378735, 0.09378735, ..., 1.0000004 , 1.0000004 ,
        0.02401601],
       [0.1179496 , 0.1179496 , 0.1179496 , ..., 0.02401601, 0.02401601,
        1.0000002 ]], shape=(52, 52), dtype=float32)

In [7]:
article_ids = events_df["article_id"].to_numpy()

different_article_mask = (
    article_ids[:, None] != article_ids[None, :]
)

cross_article_similarity = similarity_matrix * different_article_mask

print("Cross-article comparisons prepared.")
print("Mask shape:", different_article_mask.shape)
print("Similarity shape:", cross_article_similarity.shape)

Cross-article comparisons prepared.
Mask shape: (42, 42)
Similarity shape: (42, 42)


In [8]:
threshold = 0.50

pairs = []

for i in range(len(events_df)):
    for j in range(i + 1, len(events_df)):

        if article_ids[i] == article_ids[j]:
            continue

        score = float(similarity_matrix[i, j])

        if score >= threshold:

            pairs.append({
                "event_1_index": i,
                "event_2_index": j,
                "article_1": str(events_df.iloc[i]["article_id"]),
                "article_2": str(events_df.iloc[j]["article_id"]),
                "event_1": str(events_df.iloc[i]["original_sentence"]),
                "event_2": str(events_df.iloc[j]["original_sentence"]),
                "semantic_similarity": score
            })

candidate_pairs = pd.DataFrame(pairs)

if not candidate_pairs.empty:
    candidate_pairs = candidate_pairs.sort_values(
        by="semantic_similarity",
        ascending=False
    ).reset_index(drop=True)

print("Potential matching pairs:", len(candidate_pairs))

display(candidate_pairs.head(15))

Potential matching pairs: 65


,event_1_index,event_2_index,article_1,article_2,event_1,event_2,semantic_similarity
0,7,20,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079
1,7,21,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079
2,3,31,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799
3,3,30,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799
4,21,32,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379
5,20,32,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379
6,7,32,article_01,article_03,The municipal corporation opened two temporary...,The municipal corporation said that 60 residen...,0.829408
7,1,14,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811
8,0,14,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811
9,2,13,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811


In [9]:
def parse_entities(value):

    if pd.isna(value) or str(value).strip() == "":
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [str(x).lower().strip() for x in parsed]

        return [str(parsed).lower().strip()]

    except:
        return [str(value).lower().strip()]

In [10]:
events_df["parsed_locations"] = events_df["location_entities"].apply(
    parse_entities
)

events_df["parsed_times"] = events_df["temporal_entities"].apply(
    parse_entities
)

events_df[[
    "location_entities",
    "parsed_locations",
    "temporal_entities",
    "parsed_times"
]].head(10)

,location_entities,parsed_locations,temporal_entities,parsed_times
0,"['Riverton', 'the eastern part of the city']","[riverton, the eastern part of the city]","['August 13', 'early Friday morning']","[august 13, early friday morning]"
1,"['Riverton', 'the eastern part of the city']","[riverton, the eastern part of the city]","['August 13', 'early Friday morning']","[august 13, early friday morning]"
2,"['Riverton', 'the eastern part of the city']","[riverton, the eastern part of the city]","['August 13', 'early Friday morning']","[august 13, early friday morning]"
3,[],[],['twelve hours'],[twelve hours]
4,[],[],"['approximately 5:30 a.m.', 'August 14']","[approximately 5:30 a.m., august 14]"
5,"['the Old Bridge', 'the Riverside neighbourhood']","[the old bridge, the riverside neighbourhood]",[],[]
6,"['the Old Bridge', 'the Riverside neighbourhood']","[the old bridge, the riverside neighbourhood]",[],[]
7,['Central High School and the Riverside Commun...,[central high school and the riverside communi...,[],[]
8,[],[],['10 a.m.'],[10 a.m.]
9,[],[],['through Saturday'],[through saturday]


In [11]:
def jaccard_similarity(list1, list2):

    set1 = set(list1)
    set2 = set(list2)

    if not set1 or not set2:
        return 0.0

    return len(set1.intersection(set2)) / len(set1.union(set2))


def text_similarity(value1, value2):

    value1 = str(value1).lower().strip()
    value2 = str(value2).lower().strip()

    if not value1 or not value2 or value1 == "nan" or value2 == "nan":
        return 0.0

    return 1.0 if value1 == value2 else 0.0

In [12]:
structured_scores = []

for _, pair in candidate_pairs.iterrows():

    i = int(pair["event_1_index"])
    j = int(pair["event_2_index"])

    e1 = events_df.iloc[i]
    e2 = events_df.iloc[j]

    structured_scores.append({

        "event_1_index": i,
        "event_2_index": j,

        "action_similarity": text_similarity(
            e1["action"], e2["action"]
        ),

        "subject_similarity": text_similarity(
            e1["subject"], e2["subject"]
        ),

        "object_similarity": text_similarity(
            e1["object"], e2["object"]
        ),

        "location_similarity": jaccard_similarity(
            e1["parsed_locations"], e2["parsed_locations"]
        ),

        "temporal_similarity": jaccard_similarity(
            e1["parsed_times"], e2["parsed_times"]
        )
    })

structured_df = pd.DataFrame(structured_scores)

display(structured_df.head(10))

,event_1_index,event_2_index,action_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity
0,7,20,0.0,1.0,0.0,0.0,0.000000
1,7,21,0.0,0.0,0.0,0.0,0.000000
2,3,31,0.0,0.0,0.0,0.0,0.333333
3,3,30,0.0,0.0,1.0,0.0,0.333333
4,21,32,0.0,0.0,0.0,1.0,0.000000
5,20,32,0.0,0.0,0.0,1.0,0.000000
6,7,32,0.0,0.0,0.0,0.0,0.000000
7,1,14,0.0,0.0,0.0,0.5,0.000000
8,0,14,0.0,0.0,0.0,0.5,0.000000
9,2,13,0.0,0.0,1.0,0.5,0.000000


In [13]:
combined_df = candidate_pairs.merge(
    structured_df,
    on=["event_1_index", "event_2_index"],
    how="left"
)

combined_df["combined_similarity"] = (
    0.50 * combined_df["semantic_similarity"] +
    0.15 * combined_df["action_similarity"] +
    0.10 * combined_df["subject_similarity"] +
    0.10 * combined_df["object_similarity"] +
    0.10 * combined_df["location_similarity"] +
    0.05 * combined_df["temporal_similarity"]
)

combined_df = combined_df.sort_values(
    "combined_similarity",
    ascending=False
).reset_index(drop=True)

display(combined_df.head(15))

,event_1_index,event_2_index,article_1,article_2,event_1,event_2,semantic_similarity,action_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity,combined_similarity
0,13,27,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,1.0,0.0,1.0,1.0,0.000000,0.707638
1,1,29,article_01,article_03,Continuous rainfall began across Riverton late...,"Rainfall eased in Riverton on Saturday, August...",0.770366,1.0,0.0,1.0,0.5,0.000000,0.685183
2,3,30,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799,0.0,0.0,1.0,0.0,0.333333,0.561566
3,13,26,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
4,13,28,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
5,13,29,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
6,7,20,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079,0.0,1.0,0.0,0.0,0.000000,0.549540
7,16,39,article_02,article_03,The Old Bridge was closed to vehicles at aroun...,The Old Bridge remained closed while engineers...,0.684675,0.0,1.0,1.0,0.0,0.000000,0.542338
8,2,13,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.0,1.0,0.5,0.000000,0.535906
9,0,13,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.0,1.0,0.5,0.000000,0.535906


In [14]:
display(
    combined_df[[
        "article_1",
        "article_2",
        "event_1",
        "event_2",
        "semantic_similarity",
        "action_similarity",
        "location_similarity",
        "temporal_similarity",
        "combined_similarity"
    ]].head(20)
)

,article_1,article_2,event_1,event_2,semantic_similarity,action_similarity,location_similarity,temporal_similarity,combined_similarity
0,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,1.0,1.0,0.000000,0.707638
1,article_01,article_03,Continuous rainfall began across Riverton late...,"Rainfall eased in Riverton on Saturday, August...",0.770366,1.0,0.5,0.000000,0.685183
2,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799,0.0,0.0,0.333333,0.561566
3,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
4,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
5,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
6,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079,0.0,0.0,0.000000,0.549540
7,article_02,article_03,The Old Bridge was closed to vehicles at aroun...,The Old Bridge remained closed while engineers...,0.684675,0.0,0.0,0.000000,0.542338
8,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.5,0.000000,0.535906
9,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.5,0.000000,0.535906


In [15]:
MATCH_THRESHOLD = 0.70

combined_df["match_status"] = np.where(
    combined_df["combined_similarity"] >= MATCH_THRESHOLD,
    "Potential Match",
    "Not Matched"
)

print(combined_df["match_status"].value_counts())

match_status
Not Matched        64
Potential Match     1
Name: count, dtype: int64


In [16]:
matching_pairs = combined_df[
    combined_df["match_status"] == "Potential Match"
].copy()

non_matching_pairs = combined_df[
    combined_df["match_status"] == "Not Matched"
].copy()

print("Potential matching pairs:", len(matching_pairs))
print("Non-matching pairs:", len(non_matching_pairs))

Potential matching pairs: 1
Non-matching pairs: 64


In [17]:
print("Combined similarity statistics:\n")

print(combined_df["combined_similarity"].describe())

print("\nTop 15 similarity scores:\n")

display(
    combined_df[[
        "event_1",
        "event_2",
        "semantic_similarity",
        "action_similarity",
        "subject_similarity",
        "object_similarity",
        "location_similarity",
        "temporal_similarity",
        "combined_similarity"
    ]].sort_values(
        "combined_similarity",
        ascending=False
    ).head(15)
)

Combined similarity statistics:

count    65.000000
mean      0.427906
std       0.114241
min       0.254946
25%       0.352184
50%       0.435906
75%       0.535183
max       0.707638
Name: combined_similarity, dtype: float64

Top 15 similarity scores:



,event_1,event_2,semantic_similarity,action_similarity,subject_similarity,object_similarity,location_similarity,temporal_similarity,combined_similarity
0,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,1.0,0.0,1.0,1.0,0.000000,0.707638
1,Continuous rainfall began across Riverton late...,"Rainfall eased in Riverton on Saturday, August...",0.770366,1.0,0.0,1.0,0.5,0.000000,0.685183
2,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799,0.0,0.0,1.0,0.0,0.333333,0.561566
3,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
4,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
5,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,0.0,1.0,1.0,0.000000,0.557638
6,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079,0.0,1.0,0.0,0.0,0.000000,0.549540
7,The Old Bridge was closed to vehicles at aroun...,The Old Bridge remained closed while engineers...,0.684675,0.0,1.0,1.0,0.0,0.000000,0.542338
8,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.0,1.0,0.5,0.000000,0.535906
9,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.0,1.0,0.5,0.000000,0.535906


In [18]:
REVIEW_THRESHOLD = 0.50

review_candidates = combined_df[
    combined_df["combined_similarity"] >= REVIEW_THRESHOLD
].copy()

review_candidates = review_candidates.sort_values(
    "combined_similarity",
    ascending=False
).reset_index(drop=True)

print("Preliminary candidates:", len(review_candidates))

display(
    review_candidates[[
        "article_1",
        "article_2",
        "event_1",
        "event_2",
        "semantic_similarity",
        "action_similarity",
        "location_similarity",
        "temporal_similarity",
        "combined_similarity"
    ]]
)

Preliminary candidates: 24


,article_1,article_2,event_1,event_2,semantic_similarity,action_similarity,location_similarity,temporal_similarity,combined_similarity
0,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,1.0,1.0,0.000000,0.707638
1,article_01,article_03,Continuous rainfall began across Riverton late...,"Rainfall eased in Riverton on Saturday, August...",0.770366,1.0,0.5,0.000000,0.685183
2,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799,0.0,0.0,0.333333,0.561566
3,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
4,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
5,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.0,1.0,0.000000,0.557638
6,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079,0.0,0.0,0.000000,0.549540
7,article_02,article_03,The Old Bridge was closed to vehicles at aroun...,The Old Bridge remained closed while engineers...,0.684675,0.0,0.0,0.000000,0.542338
8,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.5,0.000000,0.535906
9,article_01,article_02,Continuous rainfall began across Riverton late...,Heavy rainfall continued in Riverton on Friday...,0.771811,0.0,0.5,0.000000,0.535906


In [19]:
def classify_event_pair(row):

    semantic = row["semantic_similarity"]
    location = row["location_similarity"]
    action = row["action_similarity"]
    subject = row["subject_similarity"]

    if semantic >= 0.80 and (location > 0 or action > 0 or subject > 0):
        return "possible_duplicate"

    elif semantic >= 0.60:
        return "related"

    else:
        return "unmatched"

In [20]:
review_candidates["event_relation"] = review_candidates.apply(
    classify_event_pair,
    axis=1
)

print(review_candidates["event_relation"].value_counts())

event_relation
related               21
possible_duplicate     3
Name: count, dtype: int64


In [28]:
display(
    review_candidates[[
        "article_1",
        "article_2",
        "event_1",
        "event_2",
        "semantic_similarity",
        "combined_similarity",
        "event_relation"
    ]].sort_values(
        "semantic_similarity",
        ascending=False
    )
)

,article_1,article_2,event_1,event_2,semantic_similarity,combined_similarity,event_relation
4,article_01,article_02,The municipal corporation opened two temporary...,The municipal corporation confirmed that tempo...,0.899079,0.549540,possible_duplicate
3,article_01,article_03,The city weather office recorded 118 millimetr...,The weather office reported that the city had ...,0.889799,0.561566,related
0,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379,0.620689,possible_duplicate
5,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379,0.520689,possible_duplicate
6,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379,0.520689,possible_duplicate
7,article_02,article_03,The municipal corporation confirmed that tempo...,The municipal corporation said that 60 residen...,0.841379,0.520689,possible_duplicate
8,article_01,article_03,The municipal corporation opened two temporary...,The municipal corporation said that 60 residen...,0.829408,0.514704,possible_duplicate
9,article_01,article_02,The weather office warned that further rainfal...,The weather office forecast more rain until Sa...,0.771700,0.510850,related
2,article_01,article_03,Continuous rainfall began across Riverton late...,"Rainfall eased in Riverton on Saturday, August...",0.770366,0.585183,related
1,article_02,article_03,Heavy rainfall continued in Riverton on Friday...,"Rainfall eased in Riverton on Saturday, August...",0.715276,0.607638,related


In [21]:
from collections import defaultdict

parent = list(range(len(events_df)))


def find(x):

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]


def union(x, y):

    root_x = find(x)
    root_y = find(y)

    if root_x != root_y:
        parent[root_y] = root_x

In [22]:
duplicate_pairs = review_candidates[
    review_candidates["event_relation"] == "possible_duplicate"
]

for _, row in duplicate_pairs.iterrows():

    i = int(row["event_1_index"])
    j = int(row["event_2_index"])

    union(i, j)


clusters = defaultdict(list)

for i in range(len(events_df)):
    clusters[find(i)].append(i)

print("Total event clusters:", len(clusters))
print("Duplicate clusters:", sum(len(v) > 1 for v in clusters.values()))
print("Unique event clusters:", sum(len(v) == 1 for v in clusters.values()))

Total event clusters: 39
Duplicate clusters: 1
Unique event clusters: 38


In [23]:
cluster_mapping = {}

for cluster_id, members in enumerate(clusters.values(), start=1):

    for event_index in members:
        cluster_mapping[event_index] = cluster_id


events_df["cluster_id"] = [
    cluster_mapping[i] for i in range(len(events_df))
]

display(
    events_df[[
        "cluster_id",
        "article_id",
        "original_sentence"
    ]].sort_values("cluster_id")
)

,cluster_id,article_id,original_sentence
0,1,article_01,Continuous rainfall began across Riverton late...
1,2,article_01,Continuous rainfall began across Riverton late...
2,3,article_01,Continuous rainfall began across Riverton late...
3,4,article_01,The city weather office recorded 118 millimetr...
4,5,article_01,Emergency services received their first major ...
5,6,article_01,Water entered ground-floor homes in the Rivers...
6,7,article_01,Water entered ground-floor homes in the Rivers...
32,8,article_03,The municipal corporation said that 60 residen...
21,8,article_02,The municipal corporation confirmed that tempo...
20,8,article_02,The municipal corporation confirmed that tempo...


In [24]:
def unique_values(series):

    values = []

    for value in series:

        if pd.isna(value):
            continue

        value = str(value).strip()

        if value and value not in values:
            values.append(value)

    return values


def merge_event_cluster(group):

    merged_event = {

        "representative_sentence": group["original_sentence"].iloc[0],

        "article_ids": unique_values(group["article_id"]),

        "source_sentences": unique_values(group["original_sentence"]),

        "event_triggers": unique_values(group["event_trigger"]),

        "actions": unique_values(group["action"]),

        "subjects": unique_values(group["subject"]),

        "objects": unique_values(group["object"]),

        "temporal_entities": unique_values(group["temporal_entities"]),

        "location_entities": unique_values(group["location_entities"]),

        "negation": group["negation"].astype(bool).any(),

        "event_count": len(group),

        "article_count": group["article_id"].nunique()
    }

    return pd.Series(merged_event)

In [25]:
merged_events = (
    events_df
    .groupby("cluster_id")
    .apply(merge_event_cluster, include_groups=False)
    .reset_index()
)

print("Original event records:", len(events_df))
print("Consolidated event records:", len(merged_events))

display(merged_events.head())

Original event records: 42
Consolidated event records: 39


,cluster_id,representative_sentence,article_ids,source_sentences,event_triggers,actions,subjects,objects,temporal_entities,location_entities,negation,event_count,article_count
0,1,Continuous rainfall began across Riverton late...,[article_01],[Continuous rainfall began across Riverton lat...,[began],[begin],[['Continuous rainfall']],[[]],"[['August 13', 'early Friday morning']]","[['Riverton', 'the eastern part of the city']]",False,1,1
1,2,Continuous rainfall began across Riverton late...,[article_01],[Continuous rainfall began across Riverton lat...,[causing],[cause],[['Continuous rainfall']],[[]],"[['August 13', 'early Friday morning']]","[['Riverton', 'the eastern part of the city']]",False,1,1
2,3,Continuous rainfall began across Riverton late...,[article_01],[Continuous rainfall began across Riverton lat...,[flood],[flood],[['several streets']],[[]],"[['August 13', 'early Friday morning']]","[['Riverton', 'the eastern part of the city']]",False,1,1
3,4,The city weather office recorded 118 millimetr...,[article_01],[The city weather office recorded 118 millimet...,[recorded],[record],[['The city weather office']],[['118 millimetres']],[['twelve hours']],[[]],False,1,1
4,5,Emergency services received their first major ...,[article_01],[Emergency services received their first major...,[received],[receive],[['Emergency services']],[['their first major flooding calls']],"[['approximately 5:30 a.m.', 'August 14']]",[[]],False,1,1


In [40]:
merged_events.to_csv(
    DATA_DIR / "merged_events.csv",
    index=False
)

print("Merged events saved successfully!")

Merged events saved successfully!


In [42]:
combined_df.to_csv(
    DATA_DIR / "event_matching_results.csv",
    index=False
)

print("Event matching results saved successfully!")

Event matching results saved successfully!
